In [1]:
import os
import re
import pandas as pd
import requests
from dotenv import load_dotenv
from time import sleep

# === Setup ===
env_path = "All_Tokens.env"
load_dotenv(env_path)
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")
token_index = 0

# === Paths ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_ci_detection_output.csv"

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}
ci_types = sorted(set(ci_patterns.values()))

# === Load full input ===
df = pd.read_csv(output_csv if os.path.exists(output_csv) else input_csv)
df['html_url'] = df['html_url'].astype(str).str.strip()

# === Initialize columns if missing ===
if 'yml_detected' not in df.columns:
    df['yml_detected'] = 'none'
if 'total_yml_files' not in df.columns:
    df['total_yml_files'] = 0
for ci in ci_types:
    if f"{ci}_count" not in df.columns:
        df[f"{ci}_count"] = 0

# === Filter: Only process valid repos not yet checked ===
to_process = df[
    (df['Valid_Repo_Step3'].str.lower() == 'yes') &
    (df['yml_detected'] == 'none')
].copy()

print(f"🔍 Starting CI file detection: {len(to_process)} repos to review.")

for i, (idx, row) in enumerate(to_process.iterrows()):
    url = row['html_url']
    print(f"🔎 [{i + 1}/{len(to_process)}] Checking: {url}")
    try:
        parts = url.rstrip('/').split('/')
        owner, repo = parts[-2], parts[-1]

        headers = {'Authorization': f'token {tokens[token_index % len(tokens)]}'}
        token_index += 1

        # === Get default branch ===
        r1 = requests.get(f"https://api.github.com/repos/{owner}/{repo}", headers=headers)
        if r1.status_code != 200:
            print(f"❌ Repo info failed: {r1.status_code}")
            df.at[idx, 'yml_detected'] = 'no'
            continue

        default_branch = r1.json().get('default_branch', 'main')

        # === Get file tree of default branch ===
        r2 = requests.get(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{default_branch}?recursive=1", headers=headers)
        if r2.status_code != 200:
            print(f"❌ File tree failed: {r2.status_code}")
            df.at[idx, 'yml_detected'] = 'no'
            continue

        files = [item['path'] for item in r2.json().get('tree', []) if item['type'] == 'blob']
        matched = []
        for f in files:
            for pattern, ci_type in ci_patterns.items():
                if re.search(pattern, f, re.IGNORECASE):
                    matched.append((f, ci_type))
                    break

        # Update detection results
        df.at[idx, 'yml_detected'] = 'yes' if matched else 'no'
        df.at[idx, 'total_yml_files'] = len(matched)

        ci_counts = {}
        for _, ci in matched:
            ci_counts[ci] = ci_counts.get(ci, 0) + 1

        for ci in ci_types:
            df.at[idx, f"{ci}_count"] = ci_counts.get(ci, 0)

        # Save after each repo
        df.to_csv(output_csv, index=False)

    except Exception as e:
        print(f"⚠️ Error on {url}: {e}")
        continue

print("\n✅ CI YAML file detection complete. Final output saved.")


ModuleNotFoundError: No module named 'pandas'